In [1]:
import pandas as pd
import joblib
from datetime import datetime, timedelta

# 1. Load the model and data
model = joblib.load('optimized_lightgbm_eta_predictor.pkl')
data = pd.read_csv('preprocessed.csv')

# 2. Prepare the features (same as during training)
features = ['current_stop_name', 'next_stop_name', 'day_of_week', 'is_holiday', 
            'is_peak_hour', 'weather_condition', 'passenger_count', 'current_speed', 
            'distance_to_next_stop', 'current_lat', 'current_lon']
X = data[features]

# Convert boolean columns to int (if needed)
X['is_holiday'] = X['is_holiday'].astype(int)
X['is_peak_hour'] = X['is_peak_hour'].astype(int)

# 3. Make predictions
predictions = model.predict(X)

# 4. Create results dataframe
results = data[['timestamp']].copy()
results['actual_eta'] = data['eta_minutes']
results['predicted_eta'] = predictions.round(2)  # Round to 2 decimal places

# 5. Add predicted arrival time (timestamp + predicted ETA)
results['timestamp'] = pd.to_datetime(results['timestamp'])
results['predicted_arrival_time'] = results['timestamp'] + pd.to_timedelta(results['predicted_eta'], unit='m')

# 6. Save to new CSV
results.to_csv('predictions_with_timestamps.csv', index=False)

# 7. Display sample results
print("Sample predictions:")
print(results.head(10))

# Optional: Calculate and display prediction error
results['error'] = results['actual_eta'] - results['predicted_eta']
print("\nPrediction Error Statistics:")
print(results['error'].describe())

C:\Users\cheng\AppData\Local\Temp\ipykernel_19288\1484185456.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['is_holiday'] = X['is_holiday'].astype(int)
C:\Users\cheng\AppData\Local\Temp\ipykernel_19288\1484185456.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['is_peak_hour'] = X['is_peak_hour'].astype(int)


Sample predictions:
            timestamp  actual_eta  predicted_eta  predicted_arrival_time
0 2024-01-01 06:00:00           4           4.13 2024-01-01 06:04:07.800
1 2024-01-01 06:05:00           1           1.00 2024-01-01 06:06:00.000
2 2024-01-01 06:10:00           1           0.97 2024-01-01 06:10:58.200
3 2024-01-01 06:17:00           1           0.35 2024-01-01 06:17:21.000
4 2024-01-01 06:23:00           3           3.13 2024-01-01 06:26:07.800
5 2024-01-01 06:30:00           4           4.33 2024-01-01 06:34:19.800
6 2024-01-01 06:36:00           0           0.01 2024-01-01 06:36:00.600
7 2024-01-01 06:42:00           3           2.33 2024-01-01 06:44:19.800
8 2024-01-01 06:47:00           0           0.15 2024-01-01 06:47:09.000
9 2024-01-01 06:54:00           1           0.98 2024-01-01 06:54:58.800

Prediction Error Statistics:
count    100000.000000
mean         -0.000344
std           0.507207
min          -5.670000
25%          -0.160000
50%           0.000000
75%      